In [1]:
import torch
import numpy as np
import torch.nn as nn
from tqdm import tqdm
from PIL import Image
from torch.optim import Adam, SGD, lr_scheduler
import matplotlib.pyplot as plt
from torchvision import transforms, models
from UNET_LIB.Unet import UNet
from utils import DiceLoss, SquarePad, SquarePad255
from Liver_Dataset import Liver_Dataset
from torch.utils.data import Dataset, Subset, DataLoader

train_total = 4000
device = 'cuda:1'
group_spec={
    'perfe':80,
    'poly+':94,
    'poly-':250,
    'rough':276,
    'bbox_msk':400,
    'sam_box':400,
    'point_msk':562,
    'MedSamBox':400
}

mult_spec={
    'perfe':[1,2,4,8,16,32,train_total/group_spec['perfe']],
    'poly+':[1,2,4,8,16,32,train_total/group_spec['poly+']],
    'poly-':[1,2,4,8,train_total/group_spec['poly-']],
    'rough':[1,2,4,8,train_total/group_spec['rough']],
    'bbox_msk':[1,2,4,8,train_total/group_spec['bbox_msk']],
    'sam_box':[1,2,4,8,train_total/group_spec['sam_box']],
    'point_msk':[1,2,4,train_total/group_spec['point_msk']],
    'MedSamBox':[1,2,4,8,train_total/group_spec['MedSamBox']]
}

epochs0 = 40
pretrained = 'ub'
assert pretrained in ['pb','fb','ub']


img_preprocess = transforms.Compose([    
#     SquarePad(),
#     transforms.Resize(512),
    transforms.RandomEqualize(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(0,1),
])

msk_preprocess = transforms.Compose([
#     SquarePad(),
#     transforms.Resize(512, \
#             interpolation = transforms.InterpolationMode.NEAREST),
])

In [2]:

num_classes = 2

for label_type in ['MedSamBox']:
    group_size = group_spec[label_type]
    DATA = Liver_Dataset('./Liver/','trainval',label_type, img_preprocess, msk_preprocess,crop=True,crop_size=352)

    for multiplicity in mult_spec[label_type]:
        
        
        model = UNet(in_channels = 1, num_classes=num_classes)
        loss_fn = nn.CrossEntropyLoss(ignore_index=255) #DiceLoss()


        model = model.to(device)
        model_name = f'UNet_L{label_type}_m{round(multiplicity)}_{pretrained}' 

        train_subset = Subset(DATA, list(range(round((group_size*multiplicity)))))
        val_subset = Subset(DATA, list(range(train_total, len(DATA))))
        train_loader = DataLoader(train_subset, batch_size=16, shuffle=True, drop_last=True,num_workers=32,pin_memory=True)
        val_loader = DataLoader(val_subset, batch_size=32, shuffle=False, drop_last=True,num_workers=16,pin_memory=True)

        epochs = round(epochs0*(1.5**np.log2(train_total/len(train_subset))))
        
        min_loss = np.inf
        fin_epoch = 0
        if pretrained == 'pb':
            optimizer = Adam(model.parameters(),lr=1e-4,weight_decay=1e-6)

        elif pretrained == 'fb':
            optimizer = SGD(model.module.classifier.parameters(),lr=1.5e-2,momentum=0.9,weight_decay=1e-3)
            scheduler = lr_scheduler.MultiStepLR(optimizer, milestones=[2000*i for i in range(1,5)], gamma=0.1)
        elif pretrained == 'ub':
            optimizer = SGD(model.parameters(),lr=1e-2,momentum=0.9,weight_decay=1e-5)
            scheduler = lr_scheduler.MultiStepLR(optimizer, milestones=[3000*i for i in range(1,4)], gamma=0.1)

        else:
            assert False

        try:
                checkpoint = torch.load(f'./model_checkpoints/{model_name}.pth')
                fin_epoch = checkpoint['fin_epoch']
                min_loss = checkpoint['min_loss']
                model.load_state_dict(checkpoint['model_state_dict'])
                optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
                print('optimizer loaded')
                scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
                print('scheduler loaded')
        except:
            if fin_epoch ==0:
                print(f'new model training {model_name}')
        pbar = tqdm(range(epochs-fin_epoch))
        for e in pbar:
            train_loss, val_loss = 0, 0


            model.train()
            for idx,(X,y) in enumerate(train_loader):

                with torch.no_grad():
                    valid_loc = y.to(device)
                    y = y.squeeze().to(device)
                optimizer.zero_grad()

                yhat = model(X.contiguous().to(device))
                loss = (loss_fn(yhat,y)*valid_loc).mean()
                loss.backward()
                optimizer.step()
                if pretrained in ['fb','ub']:
                    scheduler.step()

                train_loss += loss.item()
            train_loss /= (idx+1)

            model.eval()
            with torch.no_grad():
                class_intersect = np.zeros((num_classes,),dtype='float')
                class_union= np.zeros((num_classes,),dtype='float')
                for idx,(X,y) in enumerate(val_loader):
                    y = y.to(device).contiguous().flatten()

                    yhat = model(X.contiguous().to(device))
                    yhat_lab = torch.argmax(yhat, dim=1).flatten()
                    yhat_lab[y == 255] = 255

                    for j in range(num_classes):

                        y_bi = y == j
                        yhat_bi = yhat_lab == j
                        I = ((y_bi * yhat_bi).sum()).item()
                        U = (y_bi.sum() + yhat_bi.sum() - I).item()
                        assert I <= U
                        class_intersect[j] += I
                        class_union[j] += U

                IOUs = class_intersect/class_union
                val_loss=-np.mean(IOUs)
                pbar.set_description(f'Train loss: {train_loss}| IOU: {-val_loss}')

                #Dynamic class weight adjustment for loss function
                boost = 1/np.clip(IOUs,5e-2,1)
                boost = torch.tensor(boost/boost.sum(),dtype=torch.float).to(device)

            loss_fn = nn.CrossEntropyLoss(boost, ignore_index=255)   

            if e <10:
                continue

            if val_loss < min_loss:
                min_loss = val_loss
                to_save ={
                    'min_loss': min_loss,
                    'fin_epoch': e+fin_epoch+1,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict()}
                if pretrained in ['fb','ub']:
                    to_save['scheduler_state_dict']= scheduler.state_dict()

                torch.save(to_save, f'./model_checkpoints/{model_name}.pth')  


/tmp/ipykernel_15125/1131860383.py:40: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(f'./model_checkpoints/{model_name}.pth')


optimizer loaded
scheduler loaded


Train loss: 0.009342801496386529| IOU: 0.7644629488336965: 100%|█████████████████████████████| 9/9 [01:39<00:00, 11.08s/it]


optimizer loaded
scheduler loaded


Train loss: 0.008622435140423476| IOU: 0.7907283674952066: 100%|███████████████████████████| 13/13 [03:48<00:00, 17.61s/it]


...

Train loss: 0.00787454485660419| IOU: 0.8061387228430017: 100%|██████████████████████████████| 1/1 [00:30<00:00, 30.14s/it]


optimizer loaded
scheduler loaded


Train loss: 0.009646769874962047| IOU: 0.8124547650082392: 100%|███████████████████████████| 14/14 [12:55<00:00, 55.37s/it]


optimizer loaded
scheduler loaded


Train loss: 0.009781614057719707| IOU: 0.806850298457521: 100%|██████████████████████████████| 9/9 [10:14<00:00, 68.32s/it]


In [3]:
assert False

AssertionError: 

In [ ]:
pretrained='ub'
performance={}
num_classes=2
test_preprocess = transforms.Compose([    
#     SquarePad(),
#     transforms.Resize(512),
    transforms.ToTensor(),
    transforms.Normalize(0,1),
])

test_loader = DataLoader(Liver_Dataset('./Liver/','test','perfe', 
                                       test_preprocess, msk_preprocess,crop=True,crop_size=512), 
                         batch_size=8, shuffle=False, num_workers=8)
model = UNet(in_channels = 1, num_classes=num_classes)
model = model.to(device)

for label_type in ['MedSamBox']:
    performance[label_type]={}
    
    for multiplicity in mult_spec[label_type]:
        
        model_name = f'UNet_L{label_type}_m{round(multiplicity)}_{pretrained}' 
        checkpoint = torch.load(f'.//model_checkpoints/{model_name}.pth')
        model.load_state_dict(checkpoint['model_state_dict'])
        
        model.eval()
        print(f'Working on label_type={label_type}:{round(multiplicity)}')
        class_intersect = np.zeros((num_classes, ),dtype='float')
        class_union = np.zeros((num_classes, ),dtype='float')
    
        with torch.no_grad():
            for idx,(X,y) in enumerate(test_loader):
                y = y.to(device).contiguous().flatten()
                
                yhat = model(X.contiguous().to(device))
                yhat_lab = torch.argmax(yhat, dim=1).flatten()
                yhat_lab[y == 255] = 255

                for j in range(num_classes):

                    y_bi = y == j
                    yhat_bi = yhat_lab == j
                    I = ((y_bi * yhat_bi).sum()).item()
                    U = (y_bi.sum() + yhat_bi.sum() - I).item()
                    assert I <= U
                    class_intersect[j] += I
                    class_union[j] += U
            
        performance[label_type][multiplicity]=(class_intersect, class_union)
        
np.save(f'Liver_Inter-Union_{pretrained}2', performance)
print(performance)

In [ ]:
for key, value in performance['MedSamBox'].items():
    print(key, value[0][1]/value[1][1])

In [ ]:
for i in [1,2,4,7.117437722419929]:
    a,b = performance['point_msk'][i]
    print(a/b)

In [ ]:
#snipping result
test_preprocess = transforms.Compose([    
#     SquarePad(),
#     transforms.Resize(512),
    transforms.ToTensor(),
    transforms.Normalize(0,1),
])
num_classes=2
label_type='perfe'
test_loader = DataLoader(Liver_Dataset('/data/Liver/','test','perfe', test_preprocess, msk_preprocess,crop=True,crop_size=512), batch_size=8, shuffle=False)

for multiplicity in mult_spec[label_type]:
        
    model = UNet(in_channels = 1, num_classes=num_classes)
    model = nn.DataParallel(model).to(device)
        
    model_name = f'UNet_L{label_type}_m{round(multiplicity)}_{pretrained}'  
    try:
        checkpoint = torch.load(f'/data/model_checkpoints/{model_name}.pth')
        model.load_state_dict(checkpoint['model_state_dict'])
        print('Loaded', model_name)
    except:
        print(f'Error loading {model_name}')
        assert False

    model.eval()
    with torch.no_grad():    
        class_intersect = np.zeros((num_classes,),dtype='float')
        class_union= np.zeros((num_classes,),dtype='float')

        for idx,(X,y) in enumerate(test_loader):
            y = y.flatten()

            yhat = model(X.contiguous().to(device))
            yhat_lab = torch.argmax(yhat.cpu(), dim=1).flatten()
            skip_id = np.argwhere(y == 255)
            yhat_lab[skip_id] = 255

            for j in range(num_classes):

                y_bi = y == j
                yhat_bi = yhat_lab == j
                I = (y_bi * yhat_bi).sum()
                U = y_bi.sum() + yhat_bi.sum() - I
                assert I <= U
                class_intersect[j] += I
                class_union[j] += U

        IOUs = class_intersect/class_union
        val_loss=-np.mean(IOUs)
        print('IOU', -val_loss)
            

In [ ]:
IOUs